# Построение синтетического Q&A-датасета

Ноутбук генерирует и валидирует пары вопрос–ответ для двух EPUB-книг. Llama 3.1 8B создаёт кандидатов, Gemma 12B их проверяет; переиспользуемая логика находится в `notebooks/scripts/`.

EPUB-файлы читаются из `notebooks/epubs`, итоговые XLSX сохраняются в `notebooks/golden_set`. Перед запуском стартуйте Ollama и загрузите `llama3.1:8b` и `gemma4:12b`.

## 1. Подготовка окружения

Один раз установите зависимости командой `pip install -r requirements.txt`, затем выберите это окружение как kernel ноутбука.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks" and (ROOT / "scripts").is_dir():
    ROOT = ROOT.parent
elif not (ROOT / "notebooks" / "scripts").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "notebooks" / "scripts").is_dir():
    raise RuntimeError("Запускайте ноутбук из корня проекта или notebooks/")

NOTEBOOKS_DIR = ROOT / "notebooks"
sys.path.insert(0, str(NOTEBOOKS_DIR))

from scripts import collect_book, records_to_rows, split_epub

EPUB_DIR = NOTEBOOKS_DIR / "epubs"
GOLDEN_SET_DIR = NOTEBOOKS_DIR / "golden_set"
GOLDEN_SET_DIR.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", ROOT)
print("EPUB:", EPUB_DIR)
print("Golden set:", GOLDEN_SET_DIR)

## 2. Конфигурация

`BALANCE_TOP_UP=True` добирает недостающие корзины `category × context_mode` после обхода глав. `CONTINUE_EXISTING=True` продолжает сбор из XLSX; при `KEEP_EXISTING_FINALS=True` загруженные строки сохраняются в финальном наборе.

In [ ]:
BOOKS = {
    "osuzhden-pozhiznenno": EPUB_DIR / "osuzhden-pozhiznenno.epub",
    "dorogi-raya": EPUB_DIR / "dorogi-raya.epub",
}
GOLDEN_SET_FILES = {
    "osuzhden-pozhiznenno": GOLDEN_SET_DIR / "osuzhden-pozhiznenno.xlsx",
    "dorogi-raya": GOLDEN_SET_DIR / "dorogi-raya_finals.xlsx",
}

N_FINAL = 50
CANDIDATE_POOL_RATIO = 2.0
SINGLE_RATIO = 0.7
CHUNK_SIZE = 1500
CHUNK_OVERLAP = 300
SCATTERED_CHUNK_COUNTS = (2, 3)
SEED = 42

BALANCE_TOP_UP = True
MAX_TOPUP_ATTEMPTS = 60
CONTINUE_EXISTING = False
KEEP_EXISTING_FINALS = True
RUN_CHAPTER_PASS_WHEN_CONTINUING = False

missing = [str(path) for path in BOOKS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError("Не найдены EPUB:\n" + "\n".join(missing))

## 3. Контракт промптов

Полные промпты находятся в `scripts/question_pipeline.py`. Они разделены по ролям:

1. **Gemma — автор кандидата:** формулирует один естественный вопрос читателя, короткий прямой ответ и дословную цитату-опору.
2. **Qwen — независимый редактор:** не исправляет кандидата, а принимает или отклоняет его по явной rubric.

Обе модели проверяют, что вопрос понятен без технических слов «фрагмент / текст / контекст / чанк», не содержит двух вопросов и неоднозначных местоимений. Для `scattered` полный ответ обязан использовать все выбранные части, а `evidence` — содержать дословные опоры из каждой необходимой части.

После LLM-проверки код дополнительно отклоняет вопросы без ровно одного `?`, составные и мета-вопросы. Поэтому правила можно проверить независимо от поведения модели.

## 4. Проверка EPUB и чанкинга без вызовов Ollama

Проверяем, что обе книги читаются и разбиваются с теми же параметрами, которые будут использованы при сборе.

In [ ]:
chunk_summary = []
for book_key, epub_path in BOOKS.items():
    chunks = split_epub(
        epub_path,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunk_summary.append(
        {
            "book": book_key,
            "chapters": len({chunk.chapter_id for chunk in chunks}),
            "chunks": len(chunks),
        }
    )
pd.DataFrame(chunk_summary)

## 5. Генерация и валидация

Эта ячейка вызывает Ollama и может выполняться долго. При новом сборе пайплайн проходит главы по квотам и добирает дефицитные корзины; при продолжении загружает существующий XLSX и выполняет top-up.

In [ ]:
results = {}

for book_key, epub_path in BOOKS.items():
    existing_xlsx = GOLDEN_SET_FILES[book_key]
    continue_from = existing_xlsx if CONTINUE_EXISTING else None
    if continue_from is not None and not continue_from.is_file():
        raise FileNotFoundError(f"Нет файла для достройки: {continue_from}")

    print(f"\n=== {book_key} ===")
    results[book_key] = collect_book(
        epub_path,
        n_final=N_FINAL,
        candidate_pool_ratio=CANDIDATE_POOL_RATIO,
        single_ratio=SINGLE_RATIO,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        scattered_chunk_counts=SCATTERED_CHUNK_COUNTS,
        balance_top_up=BALANCE_TOP_UP,
        max_topup_attempts=MAX_TOPUP_ATTEMPTS,
        continue_from_xlsx=continue_from,
        keep_existing_finals=KEEP_EXISTING_FINALS,
        run_chapter_pass=(
            RUN_CHAPTER_PASS_WHEN_CONTINUING
            if continue_from is not None
            else True
        ),
        seed=SEED,
        show_progress=True,
    )

## 6. Диагностика

Перед экспортом проверяем покрытие глав, распределение финальных корзин и оставшиеся дефициты.

In [ ]:
diagnostics = {}
for book_key, result in results.items():
    diagnostics[book_key] = {
        "plan_vs_actual": pd.DataFrame(result.plan_vs_actual()),
        "final_distribution": pd.DataFrame(result.final_distribution()),
        "bucket_deficits": pd.DataFrame(result.bucket_deficits()),
    }
    print(f"\n=== {book_key}: {len(result.candidates)} candidates, {len(result.finals)} finals ===")
    display(diagnostics[book_key]["final_distribution"])
    display(diagnostics[book_key]["bucket_deficits"])
    display(diagnostics[book_key]["plan_vs_actual"])

## 7. Экспорт XLSX

XLSX сохраняет `chunk_ids`, необходимые для последующей достройки. Между первоначальным сбором и продолжением нельзя менять размер чанка и overlap.

In [ ]:
EXPORT_COLUMNS = [
    "book_title",
    "chapter_index",
    "chapter_id",
    "question",
    "answer",
    "evidence",
    "category",
    "context_mode",
    "chunk_indices",
    "global_chunk_indices",
    "chunk_ids",
    "chunks_text",
    "is_spoiler",
    "selected_text",
    "selection_anchor_chunk_id",
    "reader_anchor_chunk_id",
    "reading_progress_char",
    "sel_start",
    "sel_end",
]

exports = []
for book_key, result in results.items():
    finals = pd.DataFrame(records_to_rows(result.finals))[EXPORT_COLUMNS]
    xlsx_path = GOLDEN_SET_FILES[book_key]
    finals.to_excel(xlsx_path, index=False)
    exports.append({"book": book_key, "rows": len(finals), "xlsx": xlsx_path})

pd.DataFrame(exports)